# Day 75: Pandas Transform - apply, groupby, pivot

This notebook covers three essential pandas operations for data transformation and analysis:

1. **apply** - Apply functions to rows or columns
2. **groupby** - Group and aggregate data
3. **pivot** - Pivot tables and cross tabs

In [ ]:
import pandas as pd
import numpy as np

## Part 1: apply - Applying Functions

The `apply` method lets you apply a function along an axis of a DataFrame (rows or columns).

In [ ]:
# Create a sample student scores DataFrame
np.random.seed(42)
scores = np.random.randint(50, 101, (5, 3))
names = ['Guan Yu', 'Zhang Fei', 'Zhao Yun', 'Ma Chao', 'Huang Zhong']
courses = ['Chinese', 'Math', 'English']
df_scores = pd.DataFrame(data=scores, columns=courses, index=names)
df_scores

### 1.1 apply with lambda - Column-wise operation

Calculate the mean score for each student (apply across columns, axis=1).

In [ ]:
# Apply a lambda function to compute row-wise mean
df_scores['Average'] = df_scores.apply(lambda row: row.mean(), axis=1)
df_scores

### 1.2 apply with a custom function - Row-wise operation

Apply a function to each column to get descriptive statistics.

In [ ]:
# Define a custom function to compute range (max - min)
def score_range(series):
    return series.max() - series.min()

# Apply the function to each column (axis=0)
df_scores[courses].apply(score_range, axis=0)

### 1.3 apply with element-wise operations

Use `applymap` (or `map` in newer pandas) to apply a function element-wise.

In [ ]:
# Grade each score using apply with axis=None (element-wise in newer pandas)
def grade(score):
    if score >= 90:
        return 'A'
    elif score >= 80:
        return 'B'
    elif score >= 70:
        return 'C'
    else:
        return 'D'

df_grades = df_scores[courses].map(grade)
df_grades

### 1.4 apply returning multiple values

When `apply` returns a Series, it expands into multiple columns.

In [ ]:
# Apply a function that returns a Series with multiple results
def stats(series):
    return pd.Series({
        'mean': series.mean(),
        'std': series.std(),
        'min': series.min(),
        'max': series.max()
    })

df_scores[courses].apply(stats)

---
## Part 2: groupby - Group and Aggregate

The `groupby` method splits data into groups, applies a function, and combines the results.

In [ ]:
# Create a sample sales DataFrame
np.random.seed(0)
n = 50
df_sales = pd.DataFrame({
    'Date': pd.date_range('2020-01-01', periods=n, freq='D'),
    'Region': np.random.choice(['Shanghai', 'Beijing', 'Guangdong', 'Zhejiang'], n),
    'Channel': np.random.choice(['Tmall', 'JD', 'PDD', 'Douyin'], n),
    'Brand': np.random.choice(['BrandA', 'BrandB', 'BrandC'], n),
    'Price': np.random.randint(50, 300, n),
    'Quantity': np.random.randint(10, 100, n)
})
df_sales['Revenue'] = df_sales['Price'] * df_sales['Quantity']
df_sales.head(10)

### 2.1 Simple groupby with single aggregation

In [ ]:
# Total revenue by region
df_sales.groupby('Region').Revenue.sum()

### 2.2 groupby with multiple aggregation functions using agg

In [ ]:
# Multiple aggregations on Revenue
df_sales.groupby('Region').Revenue.agg(['sum', 'mean', 'max', 'min'])

### 2.3 groupby with named aggregations

In [ ]:
# Named aggregations for clearer output
df_sales.groupby('Region').Revenue.agg(
    Total_Revenue='sum',
    Avg_Revenue='mean',
    Max_Single='max',
    Min_Single='min'
)

### 2.4 groupby multiple columns

In [ ]:
# Total revenue by Region and Channel
df_sales.groupby(['Region', 'Channel']).Revenue.sum()

### 2.5 groupby with different aggregation per column

In [ ]:
# Different aggregations for different columns
df_sales.groupby('Region')[['Revenue', 'Quantity']].agg({
    'Revenue': 'sum',
    'Quantity': ['mean', 'max']
})

### 2.6 groupby with time-based grouping

In [ ]:
# Total revenue by month
df_sales.groupby(df_sales['Date'].dt.month).Revenue.sum()

---
## Part 3: pivot - Pivot Tables and Cross Tabs

Pivot tables reshape data by turning unique values from one column into multiple columns.

### 3.1 Basic pivot_table

In [ ]:
# Pivot table: total revenue by region
pd.pivot_table(df_sales, index='Region', values='Revenue', aggfunc='sum')

### 3.2 Pivot table with rows and columns

In [ ]:
# Add a Month column for demonstration
df_sales['Month'] = df_sales['Date'].dt.month

# Pivot table: revenue by Region (rows) and Month (columns)
pd.pivot_table(
    df_sales,
    index='Region',
    columns='Month',
    values='Revenue',
    aggfunc='sum',
    fill_value=0
)

### 3.3 Pivot table with margins (totals)

In [ ]:
# Pivot table with row and column totals
pd.pivot_table(
    df_sales,
    index='Region',
    columns='Month',
    values='Revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

### 3.4 Pivot table with multiple aggregation functions

In [ ]:
# Pivot table with multiple aggregation functions
pd.pivot_table(
    df_sales,
    index='Region',
    values='Revenue',
    aggfunc=['sum', 'mean', 'count']
)

### 3.5 Pivot table with multiple value columns

In [ ]:
# Pivot table with different aggregations for different value columns
pd.pivot_table(
    df_sales,
    index='Region',
    values=['Revenue', 'Quantity'],
    aggfunc={'Revenue': 'sum', 'Quantity': 'mean'}
)

### 3.6 Cross tabulation

In [ ]:
# Cross tab: count of transactions by Region and Channel
pd.crosstab(
    index=df_sales['Region'],
    columns=df_sales['Channel']
)

In [ ]:
# Cross tab with values and aggfunc
pd.crosstab(
    index=df_sales['Region'],
    columns=df_sales['Month'],
    values=df_sales['Revenue'],
    aggfunc='sum'
).fillna(0).astype(int)

### 3.7 Cross tab with normalized margins

In [ ]:
# Cross tab with margin totals and normalization
pd.crosstab(
    index=df_sales['Region'],
    columns=df_sales['Channel'],
    margins=True,
    margins_name='Total'
)

---
## Summary

| Operation | Method | Purpose |
|-----------|--------|---------|
| **apply** | `df.apply(func, axis)` | Apply a function row-wise or column-wise |
| **groupby** | `df.groupby(col).agg()` | Split-apply-combine on groups |
| **pivot_table** | `pd.pivot_table(df, index, columns, values, aggfunc)` | Reshape data into a spreadsheet-style table |
| **crosstab** | `pd.crosstab(index, columns)` | Frequency table from arrays/Series |